# Week 3
## 1.Work flow Continue

In [2]:
#2026-5-16
# This block is to import the result and workflow last time.
import numpy as np
import pandas as pd
from scipy.fftpack import dct, idct
from pathlib import Path
import matplotlib.pyplot as plt
from kilosort.io import load_ops

SAVE_PATH = Path('F:\\ACADEMIC') / '.test_data' / 'ZFM-02370_mini.imec0.ap.short.bin'

n_chan = 385          # channel number
N = 267 # Neuron Number
dtype = 'int16'

# Load the saved Sorting results from the local drive.
results_dir = SAVE_PATH.parent / 'kilosort4'
ops = load_ops(results_dir / 'ops.npy')
camps = pd.read_csv(results_dir / 'cluster_Amplitude.tsv', sep='\t')['Amplitude'].values
contam_pct = pd.read_csv(results_dir / 'cluster_ContamPct.tsv', sep='\t')['ContamPct'].values
chan_map =  np.load(results_dir / 'channel_map.npy')
templates =  np.load(results_dir / 'templates.npy')
chan_best = (templates**2).sum(axis=1).argmax(axis=-1)
chan_best = chan_map[chan_best]
amplitudes = np.load(results_dir / 'amplitudes.npy')
st = np.load(results_dir / 'spike_times.npy')
clu = np.load(results_dir / 'spike_clusters.npy')

fs = ops['fs']
firing_rates = np.unique(clu, return_counts=True)[1] * fs / st.max()
dshift = ops['dshift']

In [3]:
whitened_data = np.load('whitened_data.npy')  # shape: (n_samples, n_channels)
n_samples, n_channels = whitened_data.shape
print(f"Shape: {n_samples} Samples × {n_channels} Channels")

Shape: 1350000 Samples × 385 Channels


## 2. Baseline-Anchored Spike Labeling (No Hungarian Remapping)
This section evaluates compressed-data sorting using a strict baseline reference.

For each spike detected in compressed data, we find the nearest baseline spike in time.
If the time difference is within a tolerance window, the compressed spike inherits the baseline cluster label directly.
Unmatched spikes are assigned label `-1`.

This avoids cluster-to-cluster remapping and keeps baseline cluster identity as the only ground truth.

In [6]:
from pathlib import Path
import numpy as np


def extract_spike_times_1d(st_array):
    st = np.asarray(st_array)
    if st.ndim == 1:
        return st.astype(np.int64, copy=False)
    if st.ndim >= 2:
        return st[:, 0].astype(np.int64, copy=False)
    raise ValueError(f'Unsupported st shape: {st.shape}')


baseline_res = np.load('baseline_results.npy', allow_pickle=True).item()
all_eval = np.load('compression_eval_results.npy', allow_pickle=True).item()

baseline_st = extract_spike_times_1d(baseline_res['st'])
baseline_clu = np.asarray(baseline_res['clu']).astype(np.int64)

print(f'Baseline spikes: {len(baseline_st)}')
print(f'Baseline clusters: {len(np.unique(baseline_clu))}')
print(f'Available ratios: {list(all_eval.keys())}')



Baseline spikes: 137381
Baseline clusters: 267
Available ratios: [0.2, 0.1]


In [7]:
def assign_to_baseline_clusters(baseline_st, baseline_clu, test_st, tolerance=30):
    """
    Strict baseline standard:
    - Each compressed spike is matched to the nearest baseline spike in time.
    - Match accepted only if |dt| <= tolerance (samples).
    - Matched spike inherits baseline cluster label.
    - Unmatched spikes get label -1.
    """
    baseline_st = np.asarray(baseline_st, dtype=np.int64)
    baseline_clu = np.asarray(baseline_clu, dtype=np.int64)
    test_st = np.asarray(test_st, dtype=np.int64)

    labels = np.full(test_st.shape[0], -1, dtype=np.int64)
    matched_baseline_idx = np.full(test_st.shape[0], -1, dtype=np.int64)
    time_error = np.full(test_st.shape[0], np.nan, dtype=np.float32)

    if baseline_st.size == 0 or test_st.size == 0:
        return labels, matched_baseline_idx, time_error

    pos = np.searchsorted(baseline_st, test_st)

    left = np.clip(pos - 1, 0, baseline_st.size - 1)
    right = np.clip(pos, 0, baseline_st.size - 1)

    dt_left = np.abs(test_st - baseline_st[left])
    dt_right = np.abs(test_st - baseline_st[right])

    use_left = dt_left <= dt_right
    nearest_idx = np.where(use_left, left, right)
    nearest_dt = np.where(use_left, dt_left, dt_right)

    ok = nearest_dt <= tolerance
    matched_baseline_idx[ok] = nearest_idx[ok]
    labels[ok] = baseline_clu[nearest_idx[ok]]
    time_error[ok] = nearest_dt[ok].astype(np.float32)

    return labels, matched_baseline_idx, time_error


def summarize_baseline_anchored(baseline_clu, assigned_labels):
    assigned_labels = np.asarray(assigned_labels)
    matched = assigned_labels >= 0
    matched_rate = float(np.mean(matched)) if assigned_labels.size else 0.0

    unique_base = np.unique(baseline_clu)
    per_cluster_count = {}
    for cid in unique_base:
        per_cluster_count[int(cid)] = int(np.sum(assigned_labels == cid))

    return {
        'matched_rate_on_test_spikes': matched_rate,
        'n_test_spikes': int(assigned_labels.size),
        'n_matched_test_spikes': int(np.sum(matched)),
        'n_unmatched_test_spikes': int(np.sum(~matched)),
        'n_baseline_clusters_covered': int(np.sum(np.array(list(per_cluster_count.values())) > 0)),
        'per_baseline_cluster_assigned_count': per_cluster_count,
    }



In [8]:
tolerance = 30
anchored_results = {}

for ratio, payload in all_eval.items():
    ks_res = payload['ks_results']
    test_st = extract_spike_times_1d(ks_res['st'])

    assigned_labels, matched_baseline_idx, time_error = assign_to_baseline_clusters(
        baseline_st=baseline_st,
        baseline_clu=baseline_clu,
        test_st=test_st,
        tolerance=tolerance,
    )

    summary = summarize_baseline_anchored(baseline_clu, assigned_labels)

    anchored_results[ratio] = {
        'ratio': float(ratio),
        'tolerance_samples': int(tolerance),
        'test_st': test_st,
        'assigned_baseline_clu': assigned_labels,
        'matched_baseline_idx': matched_baseline_idx,
        'time_error_samples': time_error,
        'summary': summary,
    }

    print('\n' + '=' * 72)
    print(f'ratio={ratio:.2f}')
    print(f"matched(test->baseline): {summary['n_matched_test_spikes']}/{summary['n_test_spikes']} "
          f"({summary['matched_rate_on_test_spikes']:.4f})")
    print(f"unmatched test spikes: {summary['n_unmatched_test_spikes']}")
    print(f"baseline clusters covered: {summary['n_baseline_clusters_covered']}/"
          f"{len(np.unique(baseline_clu))}")

np.save('baseline_anchored_results.npy', anchored_results, allow_pickle=True)
print('\nSaved: baseline_anchored_results.npy')




ratio=0.20
matched(test->baseline): 203432/203869 (0.9979)
unmatched test spikes: 437
baseline clusters covered: 267/267

ratio=0.10
matched(test->baseline): 459910/462488 (0.9944)
unmatched test spikes: 2578
baseline clusters covered: 267/267

Saved: baseline_anchored_results.npy


In [15]:
# Optional: build spike index groups by baseline cluster for downstream analysis
ratio_list = [0.2,0.1]
for ratio_to_view in ratio_list:
    res = anchored_results[ratio_to_view]
    labels = res['assigned_baseline_clu']
    
    spikes_by_baseline_cluster = {
        int(cid): np.where(labels == cid)[0]
        for cid in np.unique(baseline_clu)
    }
    unmatched_spike_idx = np.where(labels < 0)[0]
    
    print("="*20)
    print(f'ratio {ratio_to_view:.2f}:')
    print(f'matched cluster groups: {len(spikes_by_baseline_cluster)}')
    print(f'unmatched spikes: {len(unmatched_spike_idx)}')
    sr=(len(spikes_by_baseline_cluster)/(len(spikes_by_baseline_cluster)+len(unmatched_spike_idx)))*100
    print(f"Sucessful ratio for {ratio_to_view:.2f}: {sr:.2f}%")



ratio 0.20:
matched cluster groups: 267
unmatched spikes: 437
Sucessful ratio for 0.20: 37.93%
ratio 0.10:
matched cluster groups: 267
unmatched spikes: 2578
Sucessful ratio for 0.10: 9.38%


## 3. Direct Kilosort Template-Matching with Baseline Templates 
Kilosort can be viewed as two stages:
1. Template learning (discovering units/templates from data).
2. Template matching (detecting spikes using existing templates).

Here we enforce baseline as the standard by disabling template re-learning on compressed data and reusing baseline template-related configuration.
This is stricter than post-hoc cluster remapping because detection is constrained by baseline template behavior from the start.

In [18]:
from pathlib import Path
import numpy as np
from scipy.fftpack import dct, idct

from kilosort import run_kilosort


def build_probe_from_baseline(baseline_res, n_chan):
    p0 = baseline_res['ops'].get('probe', {})
    chan_map = np.asarray(p0.get('chanMap', np.arange(n_chan))).reshape(-1).astype(np.int32)
    xc = np.asarray(p0.get('xc', baseline_res['xc'])).reshape(-1)
    yc = np.asarray(p0.get('yc', baseline_res['yc'])).reshape(-1)
    kc = np.asarray(p0.get('kcoords', np.zeros_like(chan_map))).reshape(-1)

    m = min(len(chan_map), len(xc), len(yc), len(kc), n_chan)
    if m < 1:
        raise ValueError('Invalid probe geometry after alignment.')

    return {
        'chanMap': np.arange(m, dtype=np.int32),
        'xc': xc[:m].astype(np.float32, copy=False),
        'yc': yc[:m].astype(np.float32, copy=False),
        'kcoords': kc[:m].astype(np.int32, copy=False),
    }


def build_template_matching_settings(baseline_res, n_chan, fs=30000):
    ops0 = baseline_res['ops']
    s = {
        'n_chan_bin': int(n_chan),
        'fs': fs,
        'nt': int(ops0.get('nt', 61)),
        'nt0min': int(ops0.get('nt0min', 20)),
        'batch_size': int(ops0.get('batch_size', 20000)),
        'nskip': int(ops0.get('nskip', 25)),
        'whitening_range': int(ops0.get('whitening_range', min(32, n_chan))),
        'nearest_chans': int(ops0.get('nearest_chans', min(32, n_chan))),
        'n_pcs': int(ops0.get('n_pcs', 6)),
        'Th_universal': float(ops0.get('Th_universal', 9)),
        'Th_learned': float(ops0.get('Th_learned', 6)),
        'duplicate_spike_ms': float(ops0.get('duplicate_spike_ms', 0.25)),
        'max_channel_distance': float(ops0.get('max_channel_distance', 100)),
        # Critical: do not learn new templates from compressed data
        'templates_from_data': False,
    }

    for k in ['d_min', 'd_max', 'dmin', 'dminx', 'min_template_size', 'template_sizes']:
        if k in ops0:
            s[k] = ops0[k]

    s['whitening_range'] = max(1, min(int(s['whitening_range']), n_chan))
    return s


def run_kilosort_template_matching_only(binary_path, baseline_res, n_chan, out_dir):
    settings = build_template_matching_settings(baseline_res, n_chan=n_chan, fs=30000)
    probe = build_probe_from_baseline(baseline_res, n_chan=n_chan)

    out = run_kilosort(
        settings=settings,
        probe=probe,
        filename=Path(binary_path),
        results_dir=Path(out_dir),
        do_CAR=False,
        data_dtype='float32',
    )

    if len(out) == 9:
        ops, st, clu, tF, Wall, similar_templates, is_ref, est_contam_rate, kept_spikes = out
    elif len(out) == 8:
        ops, st, clu, tF, Wall, similar_templates, is_ref, est_contam_rate = out
        kept_spikes = None
    else:
        raise ValueError(f'Unexpected run_kilosort return length: {len(out)}')

    result = {
        'ops': ops,
        'st': st,
        'clu': clu,
        'tF': tF,
        'Wall': Wall,
        'similar_templates': similar_templates,
        'is_ref': is_ref,
        'est_contam_rate': est_contam_rate,
        'kept_spikes': kept_spikes,
    }

    Path(out_dir).mkdir(parents=True, exist_ok=True)
    np.save(Path(out_dir) / 'results.npy', result, allow_pickle=True)
    return result



In [ ]:
# Run fixed-template-style matching on each compression ratio result file
# This assumes compressed/reconstructed binaries already exist from previous workflow.

baseline_res = np.load('baseline_results.npy', allow_pickle=True).item()


def infer_effective_n_chan(baseline_res, recon_bin, fallback_n_chan):
    # Prefer baseline probe chanMap length if available
    probe = baseline_res.get('ops', {}).get('probe', {})
    if 'chanMap' in probe:
        n_probe = int(np.asarray(probe['chanMap']).reshape(-1).size)
    else:
        n_probe = int(fallback_n_chan)

    # Validate against file size for float32 binary
    bytes_per_sample = 4
    file_size = Path(recon_bin).stat().st_size

    if n_probe > 0 and file_size % (bytes_per_sample * n_probe) == 0:
        return n_probe

    if fallback_n_chan > 0 and file_size % (bytes_per_sample * fallback_n_chan) == 0:
        return int(fallback_n_chan)

    raise ValueError(
        f'Cannot infer valid n_chan_bin for {recon_bin}. '
        f'file_size={file_size}, probe_n_chan={n_probe}, fallback_n_chan={fallback_n_chan}'
    )


compression_ratios = [0.2, 0.1]
fixed_match_results = {}

for ratio in compression_ratios:
    recon_bin = Path(f'whitened_recon_ratio_{ratio:.2f}.bin')
    if not recon_bin.exists():
        print(f'[skip] missing binary: {recon_bin}')
        continue

    n_chan_eff = infer_effective_n_chan(
        baseline_res=baseline_res,
        recon_bin=recon_bin,
        fallback_n_chan=int(whitened_data.shape[1]),
    )
    file_size = recon_bin.stat().st_size
    n_samples_eff = file_size // (4 * n_chan_eff)
    print(f'[check] ratio={ratio:.2f}, n_chan_bin={n_chan_eff}, samples={n_samples_eff}')

    out_dir = Path(f'ks_fixed_template_ratio_{ratio:.2f}')
    out_file = out_dir / 'results.npy'

    if out_file.exists():
        print(f'[load] {out_file}')
        res = np.load(out_file, allow_pickle=True).item()
    else:
        print(f'[run] ratio={ratio:.2f}, file={recon_bin}')
        res = run_kilosort_template_matching_only(
            binary_path=recon_bin,
            baseline_res=baseline_res,
            n_chan=n_chan_eff,
            out_dir=out_dir,
        )

    fixed_match_results[ratio] = res
    # Ratio-specific save (prevents any cross-ratio overwrite)
    np.save(f'fixed_template_matching_ratio_{ratio:.2f}.npy', res, allow_pickle=True)
    print(f"ratio={ratio:.2f}: spikes={len(res['st'])}, clusters={len(np.unique(res['clu']))}")

# Optional combined cache
np.save('fixed_template_matching_results.npy', fixed_match_results, allow_pickle=True)
print('Saved: fixed_template_matching_results.npy + ratio-specific files')



kilosort.run_kilosort: Kilosort version 4.1.7
kilosort.run_kilosort: Python version 3.11.15
kilosort.run_kilosort: ----------------------------------------
kilosort.run_kilosort: System information:
kilosort.run_kilosort: Windows-10-10.0.26200-SP0 AMD64
kilosort.run_kilosort: Intel64 Family 6 Model 186 Stepping 2, GenuineIntel
kilosort.run_kilosort: Using GPU for PyTorch computations. Specify `device` to change this.
kilosort.run_kilosort: Using CUDA device: NVIDIA GeForce RTX 2050 4.00GB
kilosort.run_kilosort: ----------------------------------------
kilosort.run_kilosort: Sorting [WindowsPath('whitened_recon_ratio_0.20.bin')]
kilosort.run_kilosort: Skipping common average reference.
kilosort.run_kilosort:  
kilosort.run_kilosort: Resource usage before sorting
kilosort.run_kilosort: ********************************************************
kilosort.run_kilosort: CPU usage:    25.30 %
kilosort.run_kilosort: Mem used:     89.40 %     |      13.98 GB
kilosort.run_kilosort: Mem avail:     

[check] ratio=0.20, n_chan_bin=383, samples=1350000
[run] ratio=0.20, file=whitened_recon_ratio_0.20.bin


kilosort.run_kilosort: Preprocessing filters computed in 3.02s; total 3.09s
kilosort.run_kilosort:  
kilosort.run_kilosort: Resource usage after preprocessing
kilosort.run_kilosort: ********************************************************
kilosort.run_kilosort: CPU usage:    26.40 %
kilosort.run_kilosort: Mem used:     90.20 %     |      14.12 GB
kilosort.run_kilosort: Mem avail:     1.53 / 15.65 GB
kilosort.run_kilosort: ------------------------------------------------------
kilosort.run_kilosort: GPU usage:    `conda install pynvml` for GPU usage
kilosort.run_kilosort: GPU memory:   60.75 %     |      2.43   /     4.00 GB
kilosort.run_kilosort: Allocated:     0.62 %     |      0.02   /     4.00 GB
kilosort.run_kilosort: Max alloc:    30.85 %     |      1.23   /     4.00 GB
kilosort.run_kilosort: ********************************************************
kilosort.run_kilosort:  
kilosort.run_kilosort: Computing drift correction.
kilosort.run_kilosort: -----------------------------------

In [ ]:
# Evaluate fixed-template matching results with strict baseline anchoring

tolerance = 30
fixed_anchored_eval = {}

for ratio, ks_res in fixed_match_results.items():
    test_st = extract_spike_times_1d(ks_res['st'])
    assigned_labels, matched_baseline_idx, time_error = assign_to_baseline_clusters(
        baseline_st=baseline_st,
        baseline_clu=baseline_clu,
        test_st=test_st,
        tolerance=tolerance,
    )

    summary = summarize_baseline_anchored(baseline_clu, assigned_labels)
    ratio_eval = {
        'assigned_baseline_clu': assigned_labels,
        'matched_baseline_idx': matched_baseline_idx,
        'time_error_samples': time_error,
        'summary': summary,
    }
    fixed_anchored_eval[ratio] = ratio_eval

    # Ratio-specific save (prevents any cross-ratio overwrite)
    np.save(f'fixed_template_anchored_eval_ratio_{ratio:.2f}.npy', ratio_eval, allow_pickle=True)

    print('\n' + '=' * 72)
    print(f'fixed-template ratio={ratio:.2f}')
    print(f"matched(test->baseline): {summary['n_matched_test_spikes']}/{summary['n_test_spikes']} "
          f"({summary['matched_rate_on_test_spikes']:.4f})")
    print(f"unmatched test spikes: {summary['n_unmatched_test_spikes']}")

# Optional combined cache
np.save('fixed_template_anchored_eval.npy', fixed_anchored_eval, allow_pickle=True)
print('Saved: fixed_template_anchored_eval.npy + ratio-specific files')

